# Text Embeddings: Exercise Solution

**Task 1**: _Implement this function according to the documentation and signature._

In [ ]:
import ollama
import numpy

def embed(text: str, embedding_model_name: str) -> numpy.ndarray:
    result = ollama.embeddings(
        model=embedding_model_name,
        prompt=text,
    )
    vec = numpy.array(result["embedding"])
    return vec

**Task 2**: _After implementing the function, use it to add an embedding column to the data_

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
import tensorflow as tf
from pandas import DataFrame, Series

In [ ]:
import tensorflow_datasets

imdb_dataset, dataset_info = tensorflow_datasets.load('imdb_reviews', with_info=True, as_supervised=True)
train_data, test_data = imdb_dataset['train'], imdb_dataset['test']

In [ ]:
def dataset_to_dataframe(dataset: tf.data.Dataset) -> DataFrame:
    """
    Convert a TensorFlow supervised dataset into a pandas DataFrame.

    Args:
        dataset (tf.data.Dataset): A TensorFlow dataset in a supervised format (text, label).

    Returns:
        pd.DataFrame: A DataFrame with columns 'text' and 'label'.
    """
    # Initialize lists to hold the data
    texts = []
    labels = []
    
    # Iterate over the dataset
    for text_tensor, label_tensor in tensorflow_datasets.as_numpy(dataset):
        texts.append(text_tensor.decode('utf-8'))
        labels.append(int(label_tensor))
    
    # Create a DataFrame
    df = DataFrame({'text': texts, 'label': labels})
    
    return df

In [ ]:
imdb_data_train = dataset_to_dataframe(imdb_dataset["train"])
imdb_data_test = dataset_to_dataframe(imdb_dataset["test"])

In [ ]:
document_data = imdb_data_train.sample(n=200)

In [ ]:
embedding_model_name = "nomic-embed-text"

In [ ]:
document_data["embedding"] = document_data["text"].progress_apply(lambda text: embed(text, embedding_model_name))
document_data

In [ ]:
import seaborn
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


def visualize_embeddings(
    embeddings, labels,
):
    """
    Visualize embeddings using T-SNE. Display a color bar if the labels are continuous
    and a legend if the labels are categorical (e.g., integers or booleans).

    Args:
    - embeddings (np.ndarray or pd.Series): A numpy array of shape (n_samples, n_features)
      containing the embeddings or a pandas Series where each element is an ndarray.
    - labels (list or np.ndarray): The labels corresponding to each embedding.
    - perplexity (int): The perplexity parameter for T-SNE, which controls the balance between
      local and global aspects of your data. Typically between 5 and 50.
    - n_iter (int): The number of iterations for optimization.
    - random_state (int): Seed for the random number generator for reproducibility.

    Returns:
    - None: Plots the T-SNE embedding.
    """
    # Check if embeddings is a pandas Series and convert if necessary
    if isinstance(embeddings, pd.Series):
        embeddings = np.vstack(embeddings.values)

    # Initialize T-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        #n_iter=1000,
    )

    # Perform T-SNE on embeddings
    tsne_results = tsne.fit_transform(embeddings)

    # Plotting
    plt.figure(figsize=(8, 6))  # Adjusted figure size
    if np.issubdtype(labels.dtype, np.integer) or np.issubdtype(labels.dtype, np.bool_):
        # Categorical labels: using a legend
        unique_labels = np.unique(labels)
        colors = plt.cm.coolwarm(
            np.linspace(0, 1, len(unique_labels))
        )  # Generate colors
        for i, label in enumerate(unique_labels):
            idx = labels == label
            plt.scatter(
                tsne_results[idx, 0],
                tsne_results[idx, 1],
                color=colors[i],
                label=str(label),
                s=10,
            )  # Smaller markers
        plt.legend(title="Label")
    else:
        # Continuous labels: using a color bar
        scatter = plt.scatter(
            tsne_results[:, 0],
            tsne_results[:, 1],
            c=labels,
            cmap="viridis",
            alpha=0.6,
            s=10,
        )
        plt.colorbar(scatter)

    plt.title("T-SNE visualization of embeddings")
    plt.xlabel("T-SNE axis 1")
    plt.ylabel("T-SNE axis 2")
    plt.show()

In [ ]:
try:
    visualize_embeddings(
        embeddings=document_data["embedding"],
        labels=document_data["label"]
    )
except KeyError:
    print("The embedding column is not there yet")